In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [2]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import xarray as xr


# -----------------------
# User settings
# -----------------------
DERIVED_MONTHLY_DIR = "/work/uc1275/u301827/02_MSE/full_midlatitude/derived_monthly_metpy"
SCRATCH_DIR         = "/scratch/u/u301827/full_midlatitude/txx_yearly"

# If your monthly files are named like: derived_YYYYMM.nc
MONTHLY_PATTERN     = os.path.join(DERIVED_MONTHLY_DIR, "derived_*.nc")

# Which months count as JJA
JJA_MONTHS = (6, 7, 8)

# Variable to define TXX
TASMAX_NAME = "tasmax"

# Variables you want sampled at the TXX day (edit as you like)
# Tip: include all your “*_at_tasmax” plus the derived diagnostics you created.
VARS_AT_TXX = [
    "tasmax",
    "q_at_tasmax", "2d_at_tasmax", "t_at_tasmax", "z_at_tasmax", "sp_at_tasmax","blh_at_tasmax",
    "mse", "mse_sat", "t_bound", "t_bound_mse","swvl1_at_tasmax",
    "TLCL", "zLCL", "pLCL",
]

# Float32 for compactness (leave ints / time vars alone)
CAST_FLOAT32 = True


# -----------------------
# Helpers: parse year+month from filename
# -----------------------
def parse_yyyymm_from_filename(path):
    """
    Accepts filenames containing YYYYMM, e.g. derived_197906.nc
    Returns (year, month) as ints.
    """
    base = os.path.basename(path)
    m = re.search(r"(\d{4})(\d{2})", base)
    if not m:
        raise ValueError(f"Could not parse YYYYMM from: {path}")
    y = int(m.group(1))
    mo = int(m.group(2))
    return y, mo


def group_jja_files_by_year(monthly_files):
    years = {}
    for f in monthly_files:
        y, mo = parse_yyyymm_from_filename(f)
        if mo in JJA_MONTHS:
            years.setdefault(y, []).append(f)

    # ensure each year is sorted by month
    for y in years:
        years[y] = sorted(years[y], key=lambda p: parse_yyyymm_from_filename(p)[1])
    return years


# -----------------------
# Step 1: merge JJA months into yearly JJA file (CDO)
# -----------------------
def cdo_merge_jja_for_year(year, files_jja, out_dir):
    """
    Creates: out_dir/jja_merged_<year>.nc
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"jja_merged_{year}.nc")

    if os.path.exists(out_path):
        print(f"[SKIP merge] {year} -> {out_path}")
        return out_path

    cdo = Cdo()
    print(f"[MERGE] {year}: {len(files_jja)} files -> {out_path}")
    # mergetime accepts a whitespace-separated list of input files
    cdo.mergetime(input=" ".join(files_jja), output=out_path, options="-O")
    return out_path

def max_t_in_days_before_txx(ds, t_var="t_dailymax", txx_idx=None, days_before=3, include_txx_day=True):
    """
    Compute, per grid cell, the maximum of ds[t_var] over the window of
    `days_before` days prior to the TXX day.

    By default, uses offsets [-days_before, ..., -1] (does NOT include the TXX day).
    If include_txx_day=True, uses offsets [-days_before, ..., 0].

    Parameters
    ----------
    ds : xr.Dataset
        Must contain 'time' and t_var with 'time' dimension.
    t_var : str
        Variable name for 500 hPa temperature (e.g. "t500", or your actual name).
    txx_idx : xr.DataArray
        Argmax indices along 'time' for tasmax (dims = spatial dims).
        If None, it will compute from ds[TASMAX_NAME].
    days_before : int
        Number of days before TXX to consider.
    include_txx_day : bool
        Whether to include the event day (lag 0).

    Returns
    -------
    t_pre_max : xr.DataArray
        Max temperature in the pre-window (dims = spatial dims).
    t_pre_max_time : xr.DataArray
        Time coordinate when that max occurs (dims = spatial dims).
    """
    if "time" not in ds.dims:
        raise ValueError("Dataset has no 'time' dimension.")
    if t_var not in ds:
        raise KeyError(f"'{t_var}' not in dataset vars: {list(ds.data_vars)}")
    if "time" not in ds[t_var].dims:
        raise ValueError(f"'{t_var}' has no 'time' dimension; dims are {ds[t_var].dims}")

    # Compute txx_idx if not provided
    if txx_idx is None:
        if TASMAX_NAME not in ds:
            raise KeyError(f"'{TASMAX_NAME}' not in dataset, cannot compute txx_idx automatically.")
        txx_idx = ds[TASMAX_NAME].argmax("time")

    # Build lags: [-days_before, ..., -1] (or include 0)
    end = 0 if include_txx_day else -1
    lags = list(range(-days_before, end + 1))

    # Collect values at each lag with safe masking (avoid negative time indices wrapping!)
    t_lag_list = []
    time_lag_list = []

    for lag in lags:
        idx_lag = txx_idx + lag  # DataArray of indices

        valid = idx_lag >= 0
        # If invalid, set to 0 to keep isel happy, then mask afterwards
        idx_safe = xr.where(valid, idx_lag, 0)

        t_lag = ds[t_var].isel(time=idx_safe).where(valid)
        time_lag = ds["time"].isel(time=idx_safe).where(valid)

        t_lag_list.append(t_lag)
        time_lag_list.append(time_lag)

    # Stack across lag and reduce
    t_stack = xr.concat(t_lag_list, dim="lag")
    t_pre_max = t_stack.max("lag", skipna=True)

    # time of max: find argmax across lag, then pick corresponding time
    # (xarray argmax returns first occurrence if ties)
    lag_of_max = t_stack.argmax("lag")
    time_stack = xr.concat(time_lag_list, dim="lag")
    t_pre_max_time = time_stack.isel(lag=lag_of_max)

    return t_pre_max, t_pre_max_time


# -----------------------
# Step 2: compute TXX day per grid cell and sample vars at that day
# -----------------------
def reduce_to_txx_day(
    ds,
    tasmax_name=TASMAX_NAME,
    vars_at_txx=VARS_AT_TXX,
    t500_var="t_dailymax",
    days_before_txx=(3, 5, 7),
    include_txx_day=True
):
    """
    ds must have a 'time' dimension and tasmax_name present.

    Returns a dataset with:
      - txx (max tasmax)
      - each var in vars_at_txx reduced to the gridcell-specific argmax time
      - txx_time_* variables storing which day was selected
      - NO 'time' dimension in the result
    """
    if "time" not in ds.dims:
        raise ValueError("Dataset has no 'time' dimension.")
    if tasmax_name not in ds:
        raise KeyError(f"'{tasmax_name}' not in dataset vars: {list(ds.data_vars)}")

    tasmax = ds[tasmax_name]

    # spatial dims = everything except time (works for lat/lon, or rgidcell, etc.)
    spatial_dims = [d for d in tasmax.dims if d != "time"]
    if not spatial_dims:
        raise ValueError(f"tasmax dims are {tasmax.dims}, expected time + spatial dims")

    # index of maximum tasmax along time, per grid cell
    txx_idx = tasmax.argmax("time")              # dims: spatial_dims
    txx_val = tasmax.max("time")                 # dims: spatial_dims

    # time coordinate at the argmax, per grid cell
    txx_time = ds["time"].isel(time=txx_idx)     # dims: spatial_dims, dtype datetime64 or cftime
    
    # -----------------------
    # NEW: Max T(500 hPa) in the 3 days prior to the TXX day (per grid cell)
    # -----------------------

    t500_premax_by_window = {}

    if t500_var in ds:
        for ndays in days_before_txx:
            t500_premax, t500_premax_time = max_t_in_days_before_txx(
                ds,
                t_var=t500_var,
                txx_idx=txx_idx,
                days_before=ndays,
                include_txx_day=include_txx_day,
            )
    
            if CAST_FLOAT32 and np.issubdtype(t500_premax.dtype, np.floating):
                t500_premax = t500_premax.astype("float32")
    
            t500_premax_by_window[ndays] = (t500_premax, t500_premax_time)

    # Convert time to convenient “kept as variable” forms
    # 1) unix seconds (int64)
    # Works well if time is datetime64; if cftime slips in, we convert via astype(str)
    if np.issubdtype(txx_time.dtype, np.datetime64):
        txx_time_ns = txx_time.astype("datetime64[s]").astype("int64")
        txx_time_unix = xr.DataArray(
            txx_time_ns.data,
            coords=txx_time.coords,
            dims=txx_time.dims,
            name="txx_time_unix",
            attrs={"long_name": "TXX day timestamp", "units": "seconds since 1970-01-01 00:00:00"},
        )
        # YYYYMMDD int
        yyyymmdd = txx_time.dt.strftime("%Y%m%d").astype("int32")
        txx_yyyymmdd = yyyymmdd.rename("txx_yyyymmdd")
        txx_yyyymmdd.attrs.update({"long_name": "TXX day as YYYYMMDD", "units": "1"})
        # day-of-year
        txx_doy = txx_time.dt.dayofyear.astype("int16").rename("txx_doy")
        txx_doy.attrs.update({"long_name": "Day-of-year of TXX day", "units": "day"})
    else:
        # fallback: store as strings + also YYYYMMDD extracted from strings
        tstr = txx_time.astype(str)
        txx_time_str = tstr.rename("txx_time_str")
        txx_time_str.attrs.update({"long_name": "TXX day timestamp (string)", "units": "1"})
        # Try YYYYMMDD from string like "YYYY-MM-DD..."
        ymd = xr.apply_ufunc(lambda s: int(s[:4] + s[5:7] + s[8:10]), tstr, vectorize=True)
        txx_yyyymmdd = ymd.astype("int32").rename("txx_yyyymmdd")
        txx_yyyymmdd.attrs.update({"long_name": "TXX day as YYYYMMDD", "units": "1"})
        # DOY not robust for arbitrary cftime calendars; omit unless you really need it
        txx_time_unix = None
        txx_doy = None

    # Build output dataset (no time dim)
    out = xr.Dataset()
    # -----------------------
    # Add pre-TXX max T(500 hPa) diagnostics
    # -----------------------
    for ndays, (t500_premax, t500_premax_time) in t500_premax_by_window.items():
        varname = f"t500_premax_{ndays}d"
        out[varname] = t500_premax
        out[varname].attrs.update({
            "long_name": f"Maximum {t500_var} in the {ndays} days prior to the TXX day",
            "units": ds[t500_var].attrs.get("units", "")
        })
    
        if np.issubdtype(t500_premax_time.dtype, np.datetime64):
            time_varname = f"t500_premax_{ndays}d_yyyymmdd"
            out[time_varname] = t500_premax_time.dt.strftime("%Y%m%d").astype("int32")
            out[time_varname].attrs.update({
                "long_name": f"Date of {ndays}-day pre-TXX max T(500 hPa)",
                "units": "1"
            })
        else:
            time_varname = f"t500_premax_{ndays}d_time_str"
            out[time_varname] = t500_premax_time.astype(str)
            out[time_varname].attrs.update({
                "long_name": f"Time of {ndays}-day pre-TXX max T(500 hPa) (string)",
                "units": "1"
            })

    
    out["txx"] = txx_val.astype("float32" if CAST_FLOAT32 else txx_val.dtype)
    out["txx"].attrs.update({"long_name": "Yearly maximum of tasmax over JJA", "units": tasmax.attrs.get("units", "")})

    # Add time variables
    if np.issubdtype(txx_time.dtype, np.datetime64):
        out["txx_time_unix"] = txx_time_unix
        out["txx_yyyymmdd"]  = txx_yyyymmdd
        out["txx_doy"]       = txx_doy
    else:
        out["txx_time_str"]  = txx_time_str
        out["txx_yyyymmdd"]  = txx_yyyymmdd

    # Sample requested variables at the TXX time (grid-cell-specific isel)
    # xarray supports vectorized indexing: da.isel(time=idx_da)
    for v in vars_at_txx:
        if v not in ds:
            continue
        da = ds[v]
        if "time" not in da.dims:
            # if it is already time-invariant, just keep it (optional)
            out[v] = da
            continue
        sampled = da.isel(time=txx_idx)
        # cast floats to float32 to save space
        if CAST_FLOAT32 and np.issubdtype(sampled.dtype, np.floating):
            sampled = sampled.astype("float32")
        out[v] = sampled

    # Copy some global attrs if useful
    out.attrs.update({
        "description": "Variables sampled at the grid-cell-specific TXX day (JJA only).",
        "tasmax_source": tasmax_name,
    })

    return out


def process_year_to_txx_file(year, merged_year_file, out_dir):
    """
    Reads merged JJA file for a year, writes: out_dir/txx_at_tasmax_<year>.nc
    """
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"txx_at_tasmax_{year}.nc")
    if os.path.exists(out_path):
        print(f"[SKIP txx] {year} -> {out_path}")
        return out_path

    print(f"[TXx ] {year}: {merged_year_file}")
    ds = xr.open_dataset(merged_year_file)

    out = reduce_to_txx_day(ds)

    # Write
    out.to_netcdf(out_path)

    ds.close()
    out.close()

    print(f"[DONE] {year} -> {out_path}")
    return out_path


# -----------------------
# Main driver
# -----------------------
def main():
    monthly_files = sorted(glob.glob(MONTHLY_PATTERN))
    if not monthly_files:
        raise FileNotFoundError(f"No files found for pattern: {MONTHLY_PATTERN}")

    years = group_jja_files_by_year(monthly_files)
    if not years:
        raise RuntimeError("No JJA files found after grouping. Check naming and JJA_MONTHS.")

    print(f"[INFO] Found {len(monthly_files)} monthly files; {len(years)} years with JJA coverage.")

    merged_dir = os.path.join(SCRATCH_DIR, "merged_jja")
    txx_dir    = os.path.join(SCRATCH_DIR, "txx_products")

    out_files = []
    for y in sorted(years.keys()):
        files_jja = years[y]
        if len(files_jja) != 3:
            print(f"[WARN] {y}: expected 3 JJA files, found {len(files_jja)} -> {files_jja}")
        merged = cdo_merge_jja_for_year(y, files_jja, merged_dir)
        out_f  = process_year_to_txx_file(y, merged, txx_dir)
        out_files.append(out_f)

    print(f"[SUMMARY] wrote {len(out_files)} yearly TXX files into: {txx_dir}")
    return out_files


if __name__ == "__main__":
    main()


[INFO] Found 258 monthly files; 86 years with JJA coverage.
[MERGE] 1940: 3 files -> /scratch/u/u301827/full_midlatitude/txx_yearly/merged_jja/jja_merged_1940.nc
[TXx ] 1940: /scratch/u/u301827/full_midlatitude/txx_yearly/merged_jja/jja_merged_1940.nc
[DONE] 1940 -> /scratch/u/u301827/full_midlatitude/txx_yearly/txx_products/txx_at_tasmax_1940.nc
[MERGE] 1941: 3 files -> /scratch/u/u301827/full_midlatitude/txx_yearly/merged_jja/jja_merged_1941.nc
[TXx ] 1941: /scratch/u/u301827/full_midlatitude/txx_yearly/merged_jja/jja_merged_1941.nc
[DONE] 1941 -> /scratch/u/u301827/full_midlatitude/txx_yearly/txx_products/txx_at_tasmax_1941.nc
[MERGE] 1942: 3 files -> /scratch/u/u301827/full_midlatitude/txx_yearly/merged_jja/jja_merged_1942.nc
[TXx ] 1942: /scratch/u/u301827/full_midlatitude/txx_yearly/merged_jja/jja_merged_1942.nc
[DONE] 1942 -> /scratch/u/u301827/full_midlatitude/txx_yearly/txx_products/txx_at_tasmax_1942.nc
[MERGE] 1943: 3 files -> /scratch/u/u301827/full_midlatitude/txx_yearly/m

In [3]:
import os
import re
import glob
import numpy as np
import xarray as xr

in_dir  = "/scratch/u/u301827/full_midlatitude/txx_yearly/txx_products/"
out_dir = "/work/uc1275/u301827/02_MSE/full_midlatitude/TXX"
os.makedirs(out_dir, exist_ok=True)

out_file = os.path.join(out_dir, "txx_at_tasmax_merged_1940_2025.nc")

files = sorted(glob.glob(os.path.join(in_dir, "txx_at_tasmax_*.nc")))
print(f"Found {len(files)} files")

year_re = re.compile(r"txx_at_tasmax_(\d{4})\.nc$")

def datetime64ns_to_epoch_seconds(da: xr.DataArray) -> xr.DataArray:
    """Convert datetime64[ns] DataArray to int64 seconds since 1970-01-01."""
    sec = da.astype("datetime64[s]").astype("int64")
    out = xr.DataArray(sec.data, coords=da.coords, dims=da.dims, name=da.name, attrs=dict(da.attrs))
    out.attrs["units"] = "seconds since 1970-01-01 00:00:00"
    out.attrs["calendar"] = "proleptic_gregorian"
    return out

datasets = []
for f in files:
    base = os.path.basename(f)
    m = year_re.match(base)
    if not m:
        raise ValueError(f"Could not parse year from filename: {base}")
    year = int(m.group(1))

    ds = xr.open_dataset(f)

    # Fix problematic 2D datetime fields BEFORE concatenation/writing
    # (they are coords/vars with dtype datetime64[ns])
    if "time" in ds.coords and np.issubdtype(ds["time"].dtype, np.datetime64):
        ds = ds.assign_coords(time=datetime64ns_to_epoch_seconds(ds["time"]).rename("time"))

    if "txx_time_unix" in ds and np.issubdtype(ds["txx_time_unix"].dtype, np.datetime64):
        ds["txx_time_unix"] = datetime64ns_to_epoch_seconds(ds["txx_time_unix"])

    # Add merge dimension
    ds = ds.expand_dims(year=[year])
    datasets.append(ds)

    print("OK:", base)

ds_merged = xr.concat(datasets, dim="year").sortby("year")

# close file handles
for ds in datasets:
    ds.close()

# Optional compression
encoding = {v: {"zlib": True, "complevel": 4} for v in ds_merged.data_vars}

ds_merged.to_netcdf(out_file, encoding=encoding)
ds_merged.close()

print(f"\nMerged file saved to:\n{out_file}")


Found 86 files
OK: txx_at_tasmax_1940.nc
OK: txx_at_tasmax_1941.nc
OK: txx_at_tasmax_1942.nc
OK: txx_at_tasmax_1943.nc
OK: txx_at_tasmax_1944.nc
OK: txx_at_tasmax_1945.nc
OK: txx_at_tasmax_1946.nc
OK: txx_at_tasmax_1947.nc
OK: txx_at_tasmax_1948.nc
OK: txx_at_tasmax_1949.nc
OK: txx_at_tasmax_1950.nc
OK: txx_at_tasmax_1951.nc
OK: txx_at_tasmax_1952.nc
OK: txx_at_tasmax_1953.nc
OK: txx_at_tasmax_1954.nc
OK: txx_at_tasmax_1955.nc
OK: txx_at_tasmax_1956.nc
OK: txx_at_tasmax_1957.nc
OK: txx_at_tasmax_1958.nc
OK: txx_at_tasmax_1959.nc
OK: txx_at_tasmax_1960.nc
OK: txx_at_tasmax_1961.nc
OK: txx_at_tasmax_1962.nc
OK: txx_at_tasmax_1963.nc
OK: txx_at_tasmax_1964.nc
OK: txx_at_tasmax_1965.nc
OK: txx_at_tasmax_1966.nc
OK: txx_at_tasmax_1967.nc
OK: txx_at_tasmax_1968.nc
OK: txx_at_tasmax_1969.nc
OK: txx_at_tasmax_1970.nc
OK: txx_at_tasmax_1971.nc
OK: txx_at_tasmax_1972.nc
OK: txx_at_tasmax_1973.nc
OK: txx_at_tasmax_1974.nc
OK: txx_at_tasmax_1975.nc
OK: txx_at_tasmax_1976.nc
OK: txx_at_tasmax_1977.

In [8]:
import xarray as xr
xr.open_dataset("/work/uc1275/u301827/02_MSE/full_midlatitude/TXX/txx_at_tasmax_merged_1940_2025.nc")

<xarray.Dataset>
Dimensions:                  (lat: 89, lon: 1280, year: 85)
Coordinates:
  * year                     (year) int64 1940 1941 1942 1943 ... 2022 2023 2024
  * lon                      (lon) float64 0.0 0.2812 0.5625 ... 359.4 359.7
  * lat                      (lat) float64 64.78 64.5 64.22 ... 40.33 40.05
    plev                     float64 ...
    time                     (year, lat, lon) datetime64[ns] ...
Data variables: (12/25)
    t500_premax_3d           (year, lat, lon) float32 ...
    t500_premax_3d_yyyymmdd  (year, lat, lon) int32 ...
    t500_premax_5d           (year, lat, lon) float32 ...
    t500_premax_5d_yyyymmdd  (year, lat, lon) int32 ...
    t500_premax_7d           (year, lat, lon) float32 ...
    t500_premax_7d_yyyymmdd  (year, lat, lon) int32 ...
    ...                       ...
    t_bound                  (year, lat, lon) float32 ...
    t_bound_mse              (year, lat, lon) float32 ...
    swvl1_at_tasmax          (year, lat, lon) float32 ...
    TLCL                     (year, lat, lon) float32 ...
    zLCL                     (year, lat, lon) float32 ...
    pLCL                     (year, lat, lon) float32 ...
Attributes:
    description:    Variables sampled at the grid-cell-specific TXX day (JJA ...
    tasmax_source:  tasmax